In [ ]:
import pandas as pd

# =========================================================
# LOAD CSV
# =========================================================
df = pd.read_csv(
    '/content/Reservoir water quality monitoring data.csv',
    encoding_errors='ignore',
    low_memory=False
)

# =========================================================
# CLEAN COLUMN NAMES
# =========================================================
df.columns = df.columns.str.strip()

# =========================================================
# CLEAN sampledate
# =========================================================
df['sampledate'] = (
    df['sampledate']
    .astype(str)
    .str.strip()
)

# =========================================================
# CONVERT DATE
# =========================================================
df['sampledate'] = pd.to_datetime(
    df['sampledate'],
    format='%d/%m/%Y %H:%M',
    errors='coerce'
)

# =========================================================
# CONVERT itemvalue TO NUMERIC
# =========================================================
df['itemvalue'] = pd.to_numeric(
    df['itemvalue'],
    errors='coerce'
)

# =========================================================
# CHECK YEARS BEFORE
# =========================================================
print("YEARS IN ORIGINAL DATA:")
print(sorted(df['sampledate'].dt.year.dropna().unique()))

# =========================================================
# REMOVE DUPLICATES
# =========================================================
df = df.drop_duplicates(
    subset=[
        'siteid',
        'sampledate',
        'samplelayer',
        'sampledepth',
        'itemengabbreviation'
    ]
)

# =========================================================
# PIVOT
# =========================================================
df_wide = df.pivot(
    index=[
        'siteid',
        'sitename',
        'damname',
        'county',
        'township',
        'twd97lon',
        'twd97lat',
        'twd97tm2x',
        'twd97tm2y',
        'sampledate',
        'samplelayer',
        'sampledepth'
    ],
    columns='itemengabbreviation',
    values='itemvalue'
).reset_index()

# =========================================================
# REMOVE COLUMN INDEX NAME
# =========================================================
df_wide.columns.name = None

# =========================================================
# CHECK YEARS AFTER
# =========================================================
print("\nYEARS AFTER PIVOT:")
print(sorted(df_wide['sampledate'].dt.year.dropna().unique()))

# =========================================================
# PREVIEW
# =========================================================
print("\nOUTPUT:")
print(df_wide.head())

# =========================================================
# SAVE
# =========================================================
output_path = '/content/Reservoir_water_quality_wide.csv'

df_wide.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'
)

print(f"\nSaved to: {output_path}")

YEARS IN ORIGINAL DATA:
[np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

YEARS AFTER PIVOT:
[np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

OUTPUT:
   siteid sitename damname county township    twd97lon   twd97lat  twd97tm2x  \
0    2075   阿公店水庫一   阿公店水庫    高雄市      NaN  120.344281  22.815112  182688.52   
1    2075   阿公店水庫一   阿公店水庫    高雄市      NaN  120.344281  22.815112  182688.52   
2    2075   阿公店水庫一   阿公店水庫    高雄市      NaN  120.344281  22.815112  182688.52   
3    2075   阿公店水庫一   阿公店水庫    高雄市      NaN  120.344281  22.815112  182688.52   
4    2075   阿公店水庫一   阿公店水庫    高雄市      NaN  120.344281  22.815112  182688.52   

    twd97tm2y          sampledate samplelayer  sampledepth  COD  Chl_a   SD  \
0  2523959.55 2020-12-02 09:35:00          表水          0.5  NaN    5.1  1.0   
1  2523959.55 2021-01-13 09:54:00          表水          0.5  4.7    3.8  1.0   
2  

In [ ]:
# =========================================================
# FILE LIST
# =========================================================

files_list = [
    "wqx_p_03_1993_2003.csv",
    "wqx_p_03_2004_2011.csv",
    "wqx_p_03_2012_2020.csv"
]

dfs = []

# =========================================================
# READ & CLEAN CSV
# =========================================================

for file in files_list:

    print(f"\nReading: {file}")

    # try utf-8 first, fallback big5
    try:
        df = pd.read_csv(file, encoding='utf-8')
    except:
        df = pd.read_csv(file, encoding='big5')

    # remove empty rows/columns
    df = df.dropna(axis=1, how='all')
    df = df.dropna(how='all')

    # clean column names
    df.columns = (
        df.columns.astype(str)
        .str.replace(r'<br\s*/?>', ' ', regex=True)
        .str.replace('\n', ' ', regex=False)
        .str.strip()
    )

    # remove duplicate columns
    df = df.loc[:, ~df.columns.duplicated()]

    print("Shape:", df.shape)

    dfs.append(df)

# =========================================================
# COMBINE
# =========================================================

combined_df = pd.concat(dfs, ignore_index=True, sort=False)

print("\nCombined Shape:")
print(combined_df.shape)

# =========================================================
# SAVE
# =========================================================

output_file = "wqx_p_03_combined_1993_2020.csv"

combined_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\nSaved: {output_file}")

# =========================================================
# DOWNLOAD
# =========================================================

files.download(output_file)

combined_df.head()


Reading: wqx_p_03_1993_2003.csv
Shape: (4054, 30)

Reading: wqx_p_03_2004_2011.csv
Shape: (5756, 35)

Reading: wqx_p_03_2012_2020.csv
Shape: (4551, 30)

Combined Shape:
(14361, 35)

Saved: wqx_p_03_combined_1993_2020.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,序號,縣市,水庫,測站,採樣日期,採樣深度(m),一般項目,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34
0,NaN,NaN,NaN,NaN,NaN,NaN,亞硝酸鹽氮<br />NO2-N<br />(mg/L),亞硝酸鹽氮<br />Nitrite-Nitrogen<br />(mg/L),化學需氧量<br />Chemical Oxygen Demand<br />(mg/L),導電度<br />Conductivity<br />(μmho/cm25℃),...,總鹼度<br />Alkalinity<br />(mg/L as CaCO3),葉綠素a<br />Chlorophyl-A<br />(μg/L),透明度<br />SD<br />(m),透明度<br />Transparency<br />(m),酸鹼值<br />pH,NaN,NaN,NaN,NaN,NaN
1,1.0,新北市,翡翠水庫,翡翠水庫一,2003/11/03,0.5,-,0.005,<4,78.5,...,22.1,1.14,-,3.9,7,NaN,NaN,NaN,NaN,NaN
2,2.0,新北市,翡翠水庫,翡翠水庫一,2003/08/18,0.5,-,0.016,<4,93.7,...,23.9,2.23,-,2.8,8.4,NaN,NaN,NaN,NaN,NaN
3,3.0,新北市,翡翠水庫,翡翠水庫一,2003/05/12,0.5,-,0.004,6.4,105,...,24.9,1.5,-,2.8,7.6,NaN,NaN,NaN,NaN,NaN
4,4.0,新北市,翡翠水庫,翡翠水庫一,2002/05/16,0.5,-,<0.001,7.2,79,...,17,1.4,-,0.8,7.3,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd

# =========================================================
# 1. LOAD BASE DATASET
# =========================================================

base_df = pd.read_csv(
    '/content/Reservoir_water_quality_wide.csv',
    encoding='utf-8-sig',
    low_memory=False
)

# =========================================================
# 2. KEEP ONLY REQUIRED COLUMNS
# =========================================================

base_cols = [
    'siteid',
    'sitename',
    'damname',
    'county',
    'township',
    'twd97lon',
    'twd97lat',
    'twd97tm2x',
    'twd97tm2y',
    'sampledate',
    'samplelayer',
    'sampledepth',
    'Chl_a'
]

base_df = base_df[base_cols]

# =========================================================
# 3. CLEAN BASE DATE
# =========================================================

base_df['sampledate'] = pd.to_datetime(
    base_df['sampledate'],
    errors='coerce'
)

# =========================================================
# 4. LOAD WQX DATASET
# =========================================================

df2 = pd.read_csv(
    '/content/wqx_p_03_combined_1993_2020.csv',
    header=[0,1],
    encoding_errors='ignore',
    low_memory=False
)

# =========================================================
# 5. FLATTEN HEADER
# =========================================================

df2.columns = [
    '_'.join(
        [str(x) for x in col if 'Unnamed' not in str(x)]
    ).strip('_')
    for col in df2.columns
]

# =========================================================
# 6. RENAME IMPORTANT COLUMNS
# =========================================================

rename_dict = {}

for c in df2.columns:

    if '縣市' in c:
        rename_dict[c] = 'county'

    elif '水庫' in c and '測站' not in c:
        rename_dict[c] = 'damname'

    elif '測站' in c:
        rename_dict[c] = 'sitename'

    elif '採樣日期' in c:
        rename_dict[c] = 'sampledate'

    elif '採樣深度' in c and 'Sampling Depth' not in c:
        rename_dict[c] = 'sampledepth'

    elif 'Chlorophyl-A' in c or '葉綠素a' in c:
        rename_dict[c] = 'Chl_a'

df2 = df2.rename(columns=rename_dict)

# =========================================================
# 7. CLEAN DATE
# =========================================================

df2['sampledate'] = pd.to_datetime(
    df2['sampledate'],
    errors='coerce',
    dayfirst=True
)

# =========================================================
# 8. CLEAN Chl_a
# =========================================================

df2['Chl_a'] = (
    df2['Chl_a']
    .astype(str)
    .str.replace('<', '', regex=False)
    .str.replace('-', '', regex=False)
    .str.strip()
)

df2['Chl_a'] = pd.to_numeric(
    df2['Chl_a'],
    errors='coerce'
)

# =========================================================
# 9. CREATE MISSING COLUMNS
# =========================================================

missing_cols = [
    'siteid',
    'township',
    'twd97lon',
    'twd97lat',
    'twd97tm2x',
    'twd97tm2y',
    'samplelayer'
]

for col in missing_cols:
    df2[col] = pd.NA

# =========================================================
# 9.5 FILL MISSING VALUES FROM BASE DATASET
# =========================================================

# reference table from base dataset
ref_cols = [
    'sitename',
    'damname',
    'county',
    'siteid',
    'township',
    'twd97lon',
    'twd97lat',
    'twd97tm2x',
    'twd97tm2y',
    'samplelayer'
]

ref_df = (
    base_df[ref_cols]
    .drop_duplicates(subset=['sitename'])
)

# merge reference info
df2 = df2.merge(
    ref_df,
    on='sitename',
    how='left',
    suffixes=('', '_ref')
)

# fill missing values
fill_cols = [
    'siteid',
    'township',
    'twd97lon',
    'twd97lat',
    'twd97tm2x',
    'twd97tm2y',
    'samplelayer'
]

for col in fill_cols:

    df2[col] = df2[col].fillna(
        df2[f'{col}_ref']
    )

# remove helper columns
df2 = df2.drop(
    columns=[f'{c}_ref' for c in fill_cols],
    errors='ignore'
)

# =========================================================
# 10. KEEP SAME STRUCTURE
# =========================================================

df2 = df2[base_cols]

# =========================================================
# 11. CONCATENATE
# =========================================================

final_df = pd.concat(
    [base_df, df2],
    ignore_index=True
)

# =========================================================
# 12. REMOVE DUPLICATES
# =========================================================

final_df = final_df.drop_duplicates()

# =========================================================
# 13. SORT
# =========================================================

final_df = final_df.sort_values(
    by='sampledate'
)

# =========================================================
# 14. SAVE
# =========================================================

output_path = '/content/Reservoir_water_quality_final.csv'

final_df.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'
)

# =========================================================
# 15. PREVIEW
# =========================================================

print(final_df.head())

print("\nFINAL SHAPE:", final_df.shape)

print(f"\nSaved to: {output_path}")

/tmp/ipykernel_2879/1314274578.py:182: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2[col] = df2[col].fillna(


      siteid sitename damname county township    twd97lon   twd97lat  \
9291  2152.0    明德水庫三    明德水庫    苗栗縣      NaN  120.884761  24.580426   
9399  2153.0   永和山水庫一   永和山水庫    苗栗縣      NaN  120.923504  24.659908   
9435  2153.0   永和山水庫一   永和山水庫    苗栗縣      NaN  120.923504  24.659908   
8703  2157.0    蘭潭水庫二    蘭潭水庫    嘉義市      NaN  120.479204  23.470807   
8645  2156.0    蘭潭水庫一    蘭潭水庫    嘉義市      NaN  120.479732  23.468085   

      twd97tm2x   twd97tm2y sampledate samplelayer  sampledepth   Chl_a  
9291  238328.59  2719310.99 1994-01-02          中水          1.0  16.300  
9399  242257.38  2728111.39 1994-01-02          中水         13.0   2.680  
9435  242257.38  2728111.39 1994-01-02          中水         25.0   0.301  
8703  196798.47  2596514.84 1994-01-02          底水         10.0   2.680  
8645  196851.32  2596213.21 1994-01-02          底水         10.0   2.550  

FINAL SHAPE: (19281, 13)

Saved to: /content/Reservoir_water_quality_final.csv


In [ ]:
import pandas as pd

# =========================================================
# LOAD FINAL DATASET
# =========================================================
df = pd.read_csv(
    '/content/Reservoir_water_quality_final.csv',
    encoding='utf-8-sig'
)

# =========================================================
# CONVERT sampledepth TO NUMERIC
# =========================================================
df['sampledepth'] = pd.to_numeric(
    df['sampledepth'],
    errors='coerce'
)

# =========================================================
# KEEP ONLY sampledepth = 0.5
# =========================================================

df = df[df['sampledepth'] == 0.5]

# =========================================================
# REMOVE ROWS WITH MISSING Chl_a
# =========================================================

df = df.dropna(subset=['Chl_a'])

# =========================================================
# RESET INDEX
# =========================================================

df = df.reset_index(drop=True)

# =========================================================
# CHECK RESULT
# =========================================================
print(df.head())

print('\nFINAL SHAPE:')
print(df.shape)

# =========================================================
# SAVE CLEANED DATASET
# =========================================================
output_path = '/content/Reservoir_water_quality_cleaned.csv'

df.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'
)

print(f'\nSaved to: {output_path}')

   siteid sitename damname county    twd97lon   twd97lat    twd97tm2x  \
0  2154.0   永和山水庫二   永和山水庫    苗栗縣  120.932496  24.654762  243167.2400   
1  2155.0   永和山水庫三   永和山水庫    苗栗縣  120.918052  24.653143  241705.1100   
2  2153.0   永和山水庫一   永和山水庫    苗栗縣  120.923504  24.659908  242257.3800   
3  2103.0    寶山水庫三    寶山水庫    新竹縣  121.044010  24.741820  254451.6100   
4  2098.0    石門水庫三    石門水庫    桃園市  121.252139  24.811444  275489.6568   

     twd97tm2y        sampledate samplelayer  sampledepth  Chl_a  
0  2727540.960  12/02/2002 00:00          表水          0.5   16.7  
1  2727362.440  12/02/2002 00:00          表水          0.5   21.4  
2  2728111.390  12/02/2002 00:00          中水          0.5   18.4  
3  2737182.290  11/11/2002 00:00          表水          0.5    2.1  
4  2744916.604  11/11/2002 00:00          表水          0.5    0.6  

FINAL SHAPE:
(6221, 12)

Saved to: /content/Reservoir_water_quality_cleaned.csv
